## 필독!!!

<h3> 여기 있는 코드는 절대 실행하지 마십시오. </h3>

눈으로만 보고 이해하시거나

복붙하셔서 실제 linux나 파이썬 환경에서 실행해 주시기 바랍니다.

이 파일은 jupyter 파일입니다.

여기 있는 코드는 모두 jupyter가 아닌 실제 파이썬 및 ROS2 환경에서 사용할 수 있는 코드로 작성하였습니다.

ROS2 코드를 jupyter에서 실행하는 방법이 없는 것은 아니나 별도의 방법이 따로 존재하기 때문에(`jupyter_ws` 참고)

여기 있는 코드를 실행하게 될 경우 일부 오류나 무한루프 등에 빠질 수 있는 위험이 있습니다.

### Action 패키지

#### 1. Server 

(py_Fibonacci/py_Fibonacci/fibonacci_action_server.py 참고.)

In [ ]:
import rclpy, time
from rclpy.node import Node
from rclpy.action import ActionServer
from action_tutorials_interfaces.action import Fibonacci

class FibonacciActionServer(Node):
    def __init__(self):
        super().__init__('fibonacci_action_server')
        self._action_server = ActionServer(self, Fibonacci, 'fibonacci', self.execute_callback)

    def execute_callback(self, goal_handle):
        self.get_logger().info('Executing goal...')
        feedback_msg = Fibonacci.Feedback()
        feedback_msg.partial_sequence = [0, 1]

        for i in range(2, goal_handle.request.order):
            feedback_msg.partial_sequence.append(
                feedback_msg.partial_sequence[i - 1] + feedback_msg.partial_sequence[i - 2]
            )
            goal_handle.publish_feedback(feedback_msg)
            self.get_logger().info(f'Published feedback: {feedback_msg.partial_sequence}')
            time.sleep(1)

        goal_handle.succeed()
        result = Fibonacci.Result()
        result.sequence = feedback_msg.partial_sequence
        return result
        
def main(args=None):
    rclpy.init(args=args)
    node = FibonacciActionServer()
    rclpy.spin(node)
    rclpy.shutdown()

Topic 및 Service 패키지 코드와 동일한 부분은 생략한다.

In [ ]:
import rclpy, time
from rclpy.node import Node

ROS2 기본 패키지 중 하나인 `rclpy`와 노드 기능들을 사용가능하게 하는 `Node`클래스 및 시간 모듈 `time`를 사용하는 코드.

In [ ]:
from rclpy.action import ActionServer

ActionServer는 ROS2에서 Action 서버를 생성하는 클래스다.

Topic, Service의 경우는 Node 클래스에 아래와 같은 함수가 포함되어 있으나, Action은 구조가 좀 더 복잡해서 이렇게 별도의 클래스가 따로 마련되어 있다.

```python
self.create_publisher(...)
self.create_subscription(...)
self.create_service(...)
self.create_client(...)
```

위 코드는 이 ActionServer를 사용하기 위해 모듈을 불러오는 코드이다.

In [ ]:
from action_tutorials_interfaces.action import Fibonacci

action_tutorials_interfaces 패키지 안 action 폴더에서 Fibonacci 파일을 사용하겠다는 뜻.

Fibonacci 파일을 열어보면 아래와 같은 내용으로 되어 있다.

(`cat /opt/ros/humble/share/action_tutorials_interfaces/action/Fibonacci.action` 로 직접 확인해도 된다.)

```bash
int32 order
---
int32[] sequence
---
int32[] partial_sequence
```

기본 내장 패키지는 아니기에, 혹여나 파일을 확인해봤는데 없다면 아래 명령어를 실행해서 설치하면 된다.

```bash
sudo apt install ros-humble-action-tutorials-interfaces
```


In [ ]:
class FibonacciActionServer(Node):

`Node` 클래스를 상속받는 Action Server 노드 클래스 `FibonacciActionServer`를 만든다.

In [ ]:
self._action_server = ActionServer(self, Fibonacci, 'fibonacci', self.execute_callback)

ROS2 Action Server를 `self._action_server`로 생성하는 코드.

`Fibonacci` : 액션 타입. 구조는 위에 나와있다.

`'fibonacci'` : 액션 이름. 클라이언트는 이 이름으로 연결해야 한다.

`self.execute_callback` : Client가 Goal을 보내고 그 Goal이 실행될 때 호출되는 함수

기본적으로 Action 통신은

```txt
Client가 Goal 전송 
        ↓
ROS2 ActionServer가 Goal을 GoalHandle 객체로 수신 및 관리
        ↓
Server가 Goal 처리 시작 
        ↓
처리 중간중간 Feedback 전송 가능
        ↓
작업 완료 후 Result 반환
```
순서로 진행된다.

또한 바로 뒤에 나오는 `execute_callback` 함수에 매개변수 `goal_handle`이 있는데

이 코드로 클라이언트로부터 온 GoalHandle 객체를 이 매개변수에 전달한다.

In [ ]:
def execute_callback(self, goal_handle):
    self.get_logger().info('Executing goal...')
    feedback_msg = Fibonacci.Feedback()
    feedback_msg.partial_sequence = [0, 1]

`feedback_msg = Fibonacci.Feedback()` : Fibonacci의 Feedback 부분, 즉 int32[] partial_sequence를 가져온다.

`feedback_msg.partial_sequence = [0, 1]` : 그 값에 [0, 1] 대입.

In [ ]:
for i in range(2, goal_handle.request.order):
    feedback_msg.partial_sequence.append(
        feedback_msg.partial_sequence[i - 1] + feedback_msg.partial_sequence[i - 2]
    )

`goal_handle.request.order` : goal_handle 객체에서 실질적으로 내용이 들어있는 request 부분에서 order 값(goal) 추출.

그 아래 코드는 feedback_msg의 partial_sequence에 마지막 두 값을 더한 값을 append하는 과정이다. (피보나치 수열 계산과정)

In [ ]:
goal_handle.publish_feedback(feedback_msg)

goal_handle 객체의 publish_feedback 함수를 이용해 feedback_msg를 전송하는 과정이다.

In [ ]:
self.get_logger().info(f'Published feedback: {feedback_msg.partial_sequence}')
time.sleep(1)

feedback_msg 내용 터미널에 출력

1초 대기. 즉 1초마다 이 작업을 반복(반복문이 끝날 때까지)

In [ ]:
goal_handle.succeed()

Goal 처리가 성공적으로 끝났다고 ROS2 Action 시스템에 알리는 코드.

참고로 실패로 끝났다고 알리려면

```python
goal_handle.abort()
```

중도중단되었다고 알리려면

```python
goal_handle.canceled()
```

를 사용하면 된다.

\#\# 참고

만약 Goal의 성공/실패 상태에 따라 코드를 만들고 싶다면 아래와 같은 방법이 있다.

(물론 서버의 경우 Goal의 상태를 정하기 때문에 이런 방식을 적용할 필요가 없지만,

클라이언트의 경우에는 아래 방식이 유용하게 사용될 수도 있다.)

In [ ]:
def get_result_callback(self, future):

    result_msg = future.result()                                # 결과 객체

    if result_msg.status == GoalStatus.STATUS_SUCCEEDED:        # 성공

        self.get_logger().info(
            f'Success: {result_msg.result.sequence}'
        )

    elif result_msg.status == GoalStatus.STATUS_ABORTED:        # 실패

        self.get_logger().info('Goal Failed')

    elif result_msg.status == GoalStatus.STATUS_CANCELED:       # 중단

        self.get_logger().info('Goal Canceled')

In [ ]:
result = Fibonacci.Result()
result.sequence = feedback_msg.partial_sequence
return result

Fibonacci 파일의 Result 부분을 클래스 형태로 result 변수에 저장한다.

그리고 sequence 부분에 feedback_msg의 sequence 값을 저장하고

이를 반환한다.

Service 통신과 마찬가지로 반환 객체를 `self._action_server`가 자동으로 클라이언트에게 전송한다.

main()함수는 구조가 완벽히 동일하므로 생략한다.

#### 2. Client

(py_Fibonacci/py_Fibonacci/fibonacci_action_client.py 참고.)

In [ ]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from action_tutorials_interfaces.action import Fibonacci

class FibonacciActionClient(Node):
    def __init__(self):
        super().__init__('fibonacci_action_client')
        self._client = ActionClient(self, Fibonacci, 'fibonacci')

    def send_goal(self, order):
        self._client.wait_for_server()
        goal_msg = Fibonacci.Goal()
        goal_msg.order = order
        self._client.send_goal_async(goal_msg, feedback_callback=self.feedback_callback
                                     ).add_done_callback(self.goal_response_callback)

    def feedback_callback(self, feedback):
        self.get_logger().info(f'Received feedback: {feedback.feedback.partial_sequence}')
        
    def goal_response_callback(self, future):
        goal_handle = future.result()
        if not goal_handle.accepted:
            self.get_logger().info('Goal rejected')
            return
        self.get_logger().info('Goal accepted')
        goal_handle.get_result_async().add_done_callback(self.get_result_callback)
        
    def get_result_callback(self, future):
        result = future.result().result
        self.get_logger().info(f'Result: {result.sequence}')
        rclpy.shutdown()
        
def main(args=None):
    rclpy.init(args=args)
    client = FibonacciActionClient()
    client.send_goal(10)
    rclpy.spin(client)

Topic, Service 패키지 코드 또는 Server 코드와 동일한 부분은 생략한다.

In [ ]:
from rclpy.action import ActionClient

ActionServer 클래스를 사용하기 위해 ActionServer 모듈을 불러왔듯이

이번엔 ActionClient 클래스를 사용하기 위해 이 코드를 사용한다.

In [ ]:
self._client = ActionClient(self, Fibonacci, 'fibonacci')

`self._client`라는 ActionClient 객체 생성.

`Fibonacci`라는 액션 파일을 사용하며

`fibonacci`라는 액션 이름으로 연결한다.

In [ ]:
self._client.wait_for_server()

`'fibonacci'`라는 이름의 Action Server가 실행될 때까지 기다린다.

(`self._client`가 `'fibonacci'`라는 액션 이름을 사용하므로)

In [ ]:
goal_msg = Fibonacci.Goal()
goal_msg.order = order

`Fibonacci`파일의 Goal 내용을 가져온다. int32 order가 되겠다.

그리고 그 내용의 order값을 send_goal 함수의 매개변수 order 값으로 넣어준다.

In [ ]:
self._client.send_goal_async(goal_msg, feedback_callback=self.feedback_callback)
                                .add_done_callback(self.goal_response_callback)

`send_goal_async(...)` : fibonacci Action Server에게 Goal을 비동기 방식으로 보내는 함수.

`goal_msg` : 보낼 Goal 객체.

`self.feedback_callback` : 서버로부터 피드백이 올 때마다 실행할 함수.

`